# Ch 12 · Forces: Passive, Springs, Bias, Actuators and Tendons

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/12-forces-actuators-tendons.ipynb)

Companion notebook for the chapter page `chapters/12-forces-actuators-tendons.html` ([page](../chapters/12-forces-actuators-tendons.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

**How this notebook works.** Every example cell is the same file as on the chapter page (`examples/12-forces-actuators-tendons/`). Cells run top to bottom in a fresh runtime. Most examples here use a handful of worlds on purpose: they check formulas against C MuJoCo. Only the parameter sweep scales with `ON_GPU`. Examples appear here in numeric order, which differs from the order on the chapter page, and the section numbers below are this notebook's own: to follow a run-note on the page, search for the example number (for example "Example 12.7b").

## 1. Bias forces and the gravity trick
`qfrc_bias` is what RNE computes: gravity plus Coriolis and centrifugal terms. The world body is given the acceleration `-gravity`, and that acceleration flows down the tree.

In [ ]:
# Example 12.1 - ex1_qfrc_bias_gravity.py
"""Example 12.1 - qfrc_bias of a free ball is m*g, and the gravity trick behind it.

smooth.py::rne computes the bias force c(q, qdot) = gravity + Coriolis/centrifugal terms. It does
not add gravity as a force on every body. Instead _cacc_world sets the acceleration of the world
body to -gravity (the world "accelerates up" by g) and the forward pass carries that acceleration
to every body. A 0.5 kg ball at rest then needs qfrc_bias[z] = m g = 4.905 N to "stay put".
The second half checks MJWarp's qfrc_bias on the shipped humanoid against C MuJoCo, and dumps the
humanoid tree (branches, subtree ranges) that drives the RNE stepper on the page.
Run: python ex1_qfrc_bias_gravity.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

BALL = """
<mujoco model="ball">
  <option timestep="0.002"/>
  <worldbody>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom type="sphere" size=".1" mass="0.5"/>
    </body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(BALL)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=2)
mjw.forward(m, d)
print("free ball, mass 0.5 kg, at rest")
print("  qfrc_bias (world 0):", d.qfrc_bias.numpy()[0])
print("  m * g =", 0.5 * 9.81)
print("  cacc of the world body (rot | lin):", d.cacc.numpy()[0, 0], "<- -gravity")
print("  cacc of the ball:                  ", d.cacc.numpy()[0, 1])

# spin the ball: a free body's bias force also holds the gyroscopic term w x (I w)
qvel = np.zeros((2, 6), dtype=np.float32)
qvel[1, 3:] = [0.0, 5.0, 5.0]                   # world 1 spins about (0, 1, 1)
d.qvel.assign(qvel)
mjw.forward(m, d)
print("world 1 spinning, qfrc_bias:", d.qfrc_bias.numpy()[1], "(sphere: I w is parallel to w)")

# humanoid: MJWarp vs C MuJoCo at a random pose and velocity
HUMANOID_PATH = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"
hm = mujoco.MjModel.from_xml_path(str(HUMANOID_PATH))
hd = mujoco.MjData(hm)
rng = np.random.default_rng(0)
hd.qpos[7:] = rng.uniform(-0.3, 0.3, hm.nq - 7)
hd.qvel[:] = rng.uniform(-1.0, 1.0, hm.nv)
mujoco.mj_forward(hm, hd)
hmw = mjw.put_model(hm)
hdw = mjw.put_data(hm, hd, nworld=1)
mjw.forward(hmw, hdw)
err = np.abs(hdw.qfrc_bias.numpy()[0] - hd.qfrc_bias).max()
print(f"humanoid nv={hm.nv}, nbranch={hmw.nbranch}: max |qfrc_bias MJWarp - C| = {err:.2e}")

if args.dump_json:
    start = hmw.body_branch_start.numpy()
    flat = hmw.body_branches.numpy()
    out = {"names": [hm.body(i).name for i in range(hm.nbody)],
           "parent": hm.body_parentid.tolist(),
           "branches": [flat[start[b]:start[b + 1]].tolist() for b in range(hmw.nbranch)],
           "subtreenum": hmw.body_subtreenum.numpy().tolist(),
           "dofnum": hm.body_dofnum.tolist()}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 2. Passive forces by name
Springs, dampers (including MuJoCo 4 polynomial terms), gravcomp and fluid forces are written to separate arrays and summed into `qfrc_passive`. Note `<compiler angle="radian"/>`: without it `springref` is read in degrees.

In [ ]:
# Example 12.2 - ex2_spring_damper.py
"""Example 12.2 - joint springs and dampers, polynomial stiffness, and qfrc_passive by name.

passive.py::_spring_damper_dof_passive (one thread per (world, joint)) writes qfrc_spring and
qfrc_damper. For a hinge: qfrc_spring = -x * (k + p1*x + p2*x^2) with x = q - springref, and
qfrc_damper = -v * (b + p1*|v| + p2*v^2) (util_misc._poly_force; flg_odd=1 uses |v|). MuJoCo 4
reads the polynomial terms from extra numbers in stiffness="k p1 p2" and damping="b p1 p2".
Part (c) builds a 3-link arm with springs, dampers, gravcomp and fluid drag and checks that
qfrc_passive = qfrc_spring + qfrc_damper + qfrc_gravcomp + qfrc_fluid (+ qfrc_adhesion, 0 here;
flex passive forces are written into qfrc_spring / qfrc_damper).
Run: python ex2_spring_damper.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

HINGE = """
<mujoco model="spring_hinge">
  <compiler angle="radian"/>  <!-- springref is an angle: without this it is read in degrees -->
  <option gravity="0 0 0"/>
  <worldbody>
    <body name="flap">
      <joint name="j" type="hinge" axis="0 1 0" stiffness="{stiffness}" damping="{damping}"
             springref="0.2"/>
      <geom type="box" size=".2 .02 .05" pos=".2 0 0" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""


def passive_at(xml, qpos, qvel):
    """One world per (qpos, qvel) pair; returns MJWarp and C MuJoCo qfrc_spring, qfrc_damper."""
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=len(qpos))
    d.qpos.assign(np.array(qpos, dtype=np.float32)[:, None])
    d.qvel.assign(np.array(qvel, dtype=np.float32)[:, None])
    mjw.forward(m, d)
    c_spring, c_damper = [], []
    for q, v in zip(qpos, qvel):
        md = mujoco.MjData(mjm)
        md.qpos[0], md.qvel[0] = q, v
        mujoco.mj_forward(mjm, md)
        c_spring.append(md.qfrc_spring[0])
        c_damper.append(md.qfrc_damper[0])
    return (mjm, d.qfrc_spring.numpy()[:, 0], d.qfrc_damper.numpy()[:, 0],
            np.array(c_spring), np.array(c_damper))


Q = [0.0, 0.5, -0.4]
V = [1.0, -2.0, 0.5]
REF = 0.2

# (a) linear spring and damper
mjm, sp, dm, csp, cdm = passive_at(HINGE.format(stiffness="10", damping="0.5"), Q, V)
print("(a) linear: k=10, b=0.5, springref=0.2")
print("  jnt_stiffness", mjm.jnt_stiffness, "| jnt_stiffnesspoly", mjm.jnt_stiffnesspoly)
for q, v, s, dd in zip(Q, V, sp, dm):
    print(f"  q={q:+.2f} v={v:+.2f}: qfrc_spring={s:+.4f} (formula {-10 * (q - REF):+.4f})"
          f"  qfrc_damper={dd:+.4f} (formula {-0.5 * v:+.4f})")

# (b) MuJoCo 4 polynomial stiffness and damping: stiffness="k p1 p2", damping="b p1 p2"
K, KP = 10.0, (5.0, 40.0)
B, BP = 0.5, (0.2, 0.1)
mjm, sp, dm, csp, cdm = passive_at(
    HINGE.format(stiffness=f"{K} {KP[0]} {KP[1]}", damping=f"{B} {BP[0]} {BP[1]}"), Q, V)
print("(b) polynomial: stiffness='10 5 40', damping='0.5 0.2 0.1'")
print("  jnt_stiffnesspoly", mjm.jnt_stiffnesspoly, "| dof_dampingpoly", mjm.dof_dampingpoly)
for q, v, s, dd, cs, cd in zip(Q, V, sp, dm, csp, cdm):
    x = q - REF
    f_s = -x * (K + KP[0] * x + KP[1] * x * x)
    f_d = -v * (B + BP[0] * abs(v) + BP[1] * v * v)
    print(f"  x={x:+.2f}: spring {s:+.4f} numpy {f_s:+.4f} C {cs:+.4f} | "
          f"v={v:+.2f}: damper {dd:+.4f} numpy {f_d:+.4f} C {cd:+.4f}")

# (c) qfrc_passive decomposed by name on a 3-link arm
ARM = """
<mujoco model="passive_arm">
  <compiler angle="radian"/>
  <option timestep="0.002" density="1.2" viscosity="0.00002"/>
  <worldbody>
    <body name="l1" pos="0 0 1">
      <joint name="j1" type="hinge" axis="0 1 0" stiffness="4" damping="0.4"/>
      <geom type="capsule" fromto="0 0 0 .4 0 0" size=".05" mass="1"/>
      <body name="l2" pos=".4 0 0">
        <joint name="j2" type="hinge" axis="0 1 0" stiffness="2 1 0" damping="0.2" springref="0.3"/>
        <geom type="capsule" fromto="0 0 0 .3 0 0" size=".04" mass="0.6"/>
        <body name="l3" pos=".3 0 0" gravcomp="1">
          <joint name="j3" type="hinge" axis="0 1 0" damping="0.1"/>
          <geom type="box" size=".15 .1 .02" pos=".15 0 0" mass="0.3"/>
        </body>
      </body>
    </body>
  </worldbody>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(ARM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)
d.qpos.assign(np.array([[0.4, -0.5, 0.8]], dtype=np.float32))
d.qvel.assign(np.array([[1.5, -3.0, 6.0]], dtype=np.float32))
mjw.forward(m, d)
parts = {k: getattr(d, k).numpy()[0] for k in
         ("qfrc_spring", "qfrc_damper", "qfrc_gravcomp", "qfrc_fluid", "qfrc_passive")}
print("(c) 3-link arm, m.has_fluid =", m.has_fluid)
for k, v in parts.items():
    print(f"  {k:14s}", v)
resid = parts["qfrc_passive"] - (parts["qfrc_spring"] + parts["qfrc_damper"]
                                 + parts["qfrc_gravcomp"] + parts["qfrc_fluid"])
print("  passive - (spring + damper + gravcomp + fluid) =", resid)
md = mujoco.MjData(mjm)
md.qpos[:], md.qvel[:] = [0.4, -0.5, 0.8], [1.5, -3.0, 6.0]
mujoco.mj_forward(mjm, md)
err = np.abs(parts["qfrc_passive"] - md.qfrc_passive).max()
print(f"  max |qfrc_passive MJWarp - C| = {err:.2e}")

if args.dump_json:
    out = {"dofs": ["j1", "j2", "j3"],
           **{k: [round(float(x), 4) for x in v] for k, v in parts.items()}}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Stacked bars: qfrc_passive of the 3-link arm by name (uses `parts` from Example 12.2).
try:
    import matplotlib.pyplot as plt
    names = ["qfrc_spring", "qfrc_damper", "qfrc_gravcomp", "qfrc_fluid"]
    pos, neg = np.zeros(3), np.zeros(3)
    fig, ax = plt.subplots(figsize=(5, 3.2))
    for k, n in enumerate(names):
        v = np.array(parts[n])
        c = f"C{k}"
        ax.bar(range(3), np.where(v > 0, v, 0), bottom=pos, color=c, label=n)
        ax.bar(range(3), np.where(v < 0, v, 0), bottom=neg, color=c)
        pos += np.where(v > 0, v, 0)
        neg += np.where(v < 0, v, 0)
    ax.scatter(range(3), parts["qfrc_passive"], color="k", zorder=3, label="qfrc_passive")
    ax.set_xticks(range(3), ["j1", "j2", "j3"])
    ax.axhline(0, color="k", lw=0.8)
    ax.legend(fontsize=7)
    plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

### Plastic springs
With `springyield` the spring reference lives in `d.plastic` and moves after every step (`forward.py::_advance` → `_next_plastic_joint`).

In [ ]:
# Example 12.2b - ex2b_plastic_spring.py
"""Example 12.2b - a plastic spring: the spring reference moves when the joint yields.

MuJoCo 4 joints and tendons can have springyield: when |q - ref| exceeds the yield distance, the
spring reference is dragged along so that |q - ref| = yield (a "return mapping"). The reference
lives in d.plastic (m.jnt_plasticadr[j] >= 0 is the joint's slot) and is updated after every step
in forward.py::_advance by _next_plastic_joint. passive.py::_spring_damper_dof_passive then reads
the spring reference from d.plastic instead of m.qpos_spring.
We bend a plastic hinge with a large qfrc_applied, release it, and compare d.plastic and qpos with
C MuJoCo step by step (parity check on CPU).
Run: python ex2b_plastic_spring.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

PLASTIC = """
<mujoco model="plastic_hinge">
  <compiler angle="radian"/>
  <option timestep="0.002" gravity="0 0 0"/>
  <worldbody>
    <body name="wire">
      <joint name="bend" type="hinge" axis="0 1 0" stiffness="20" springref="0"
             springyield="0.1" damping="1"/>
      <geom type="capsule" fromto="0 0 0 .3 0 0" size=".02" mass="0.2"/>
    </body>
  </worldbody>
</mujoco>
"""
NSTEP = 1500        # 3 s: push for 0.5 s, then release
PUSH = 3.0          # N m, held for the first 250 steps

mjm = mujoco.MjModel.from_xml_string(PLASTIC)
print("nplastic =", mjm.nplastic, "| jnt_plasticadr =", mjm.jnt_plasticadr,
      "| jnt_springyield =", mjm.jnt_springyield)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)
md = mujoco.MjData(mjm)

frames, maxdiff = [], 0.0
for i in range(NSTEP):
    tau = PUSH if i < 250 else 0.0
    d.qfrc_applied.fill_(tau)
    md.qfrc_applied[0] = tau
    mjw.step(m, d)
    mujoco.mj_step(mjm, md)
    q, ref = float(d.qpos.numpy()[0, 0]), float(d.plastic.numpy()[0, 0])
    maxdiff = max(maxdiff, abs(q - md.qpos[0]), abs(ref - md.plastic[0]))
    if i % 25 == 24:
        frames.append([round((i + 1) * mjm.opt.timestep, 3), round(q, 4), round(ref, 4)])
    if i + 1 in (50, 250, 1500):
        print(f"t={(i + 1) * mjm.opt.timestep:.1f}s q={q:+.4f} plastic ref={ref:+.4f} "
              f"(C: q={md.qpos[0]:+.4f} ref={md.plastic[0]:+.4f}) q-ref={q - ref:+.4f}")
print(f"elastic limit would hold q at PUSH/k = {PUSH / 20:.3f}; yield = 0.1")
print(f"max |MJWarp - C| over {NSTEP} steps (qpos and plastic): {maxdiff:.2e}")
print("after release the wire rests at its new reference: permanent set =",
      round(float(d.plastic.numpy()[0, 0]), 4), "rad")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"push": PUSH, "k": 20, "yield": 0.1, "frames": frames}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 3. Actuators: one formula
`force = gain * ctrl + bias`, clamped to `forcerange`.

In [ ]:
# Example 12.3 - ex3_actuator_types.py
"""Example 12.3 - motor, position and velocity actuators are one formula with different numbers.

forward.py::_actuator_force (one thread per (world, actuator)) computes
    force = gain * ctrl_act + bias,  gain = gainprm[0] (+ gainprm[1]*length + gainprm[2]*velocity)
                                     bias = biasprm[0] + biasprm[1]*length + biasprm[2]*velocity
then clamps it to forcerange. The MJCF shortcuts only fill in gainprm/biasprm:
  <motor>    gain = 1,              bias = 0                    -> force = ctrl
  <position> gain = kp,             bias = -kp*length - kv*vel  -> force = kp*(ctrl - q) - kv*qdot
  <velocity> gain = kv,             bias = -kv*velocity         -> force = kv*(ctrl - qdot)
Run: python ex3_actuator_types.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

THREE = """
<mujoco model="three_actuators">
  <option gravity="0 0 0"/>
  <worldbody>
    <body>
      <joint name="j" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .3 0 0" size=".03" mass="1"/>
    </body>
  </worldbody>
  <actuator>
    <motor    name="motor" joint="j" gear="1"/>
    <position name="pos"   joint="j" kp="20"/>
    <velocity name="vel"   joint="j" kv="3" forcerange="-2 2"/>
  </actuator>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(THREE)
for i in range(mjm.nu):
    a = mjm.actuator(i)
    print(f"{a.name:5s} gaintype={mujoco.mjtGain(a.gaintype[0]).name:12s} "
          f"biastype={mujoco.mjtBias(a.biastype[0]).name:12s} "
          f"gainprm[:3]={a.gainprm[:3]} biasprm[:3]={a.biasprm[:3]}")

m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=3)
# three worlds: (qpos, qvel) states; every world sends ctrl = [0.7, 0.5, 1.0]
d.qpos.assign(np.array([[0.0], [0.2], [-0.1]], dtype=np.float32))
d.qvel.assign(np.array([[0.0], [0.4], [-2.0]], dtype=np.float32))
d.ctrl.assign(np.tile(np.array([0.7, 0.5, 1.0], dtype=np.float32), (3, 1)))
mjw.forward(m, d)

force = d.actuator_force.numpy()
for w, (q, v) in enumerate([(0.0, 0.0), (0.2, 0.4), (-0.1, -2.0)]):
    expect = [0.7, 20 * (0.5 - q), float(np.clip(3 * (1.0 - v), -2, 2))]
    print(f"world {w}: q={q:+.1f} qdot={v:+.1f}  actuator_force={force[w]}  "
          f"formula={np.array(expect)}")
print("qfrc_actuator (sum over actuators, moment = gear = 1):", d.qfrc_actuator.numpy()[:, 0])
print("actuator_length = q * gear:", d.actuator_length.numpy()[:, 0])

## 4. Activation dynamics
A filter actuator has an internal state `act`.

In [ ]:
# Example 12.4 - ex4_filter_dynamics.py
"""Example 12.4 - actuator activation dynamics: a first-order filter on ctrl.

With dyntype="filter" the actuator has one activation state act (stored in d.act) with
    act_dot = (ctrl - act) / tau          (forward.py::_actuator_force, tau = dynprm[0])
and the force uses act instead of ctrl. After a step input ctrl = 1 the exact answer is
act(t) = 1 - exp(-t / tau). "filter" integrates act with the integrator (Euler here),
"filterexact" uses the exact exponential update (support.py::next_act). actearly="true" lets the
force use the act of the *next* step, which removes one step of delay.
Run: python ex4_filter_dynamics.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

FILTERS = """
<mujoco model="filters">
  <option timestep="0.01" gravity="0 0 0"/>
  <worldbody>
    <body>
      <joint name="j" type="slide" axis="1 0 0"/>
      <geom type="sphere" size=".05" mass="1"/>
    </body>
  </worldbody>
  <actuator>
    <general name="filter"      joint="j" dyntype="filter"      dynprm="0.1" gainprm="1"/>
    <general name="filterexact" joint="j" dyntype="filterexact" dynprm="0.1" gainprm="1"/>
    <general name="early"       joint="j" dyntype="filterexact" dynprm="0.1" gainprm="1"
             actearly="true"/>
  </actuator>
</mujoco>
"""
TAU = 0.1
mjm = mujoco.MjModel.from_xml_string(FILTERS)
print(f"nu={mjm.nu}, na={mjm.na}, actuator_actadr={mjm.actuator_actadr}, "
      f"actuator_actnum={mjm.actuator_actnum}")
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)
d.ctrl.fill_(1.0)                              # step input at t = 0

mjw.forward(m, d)
print("before any step: act =", d.act.numpy()[0], "force =", d.actuator_force.numpy()[0],
      "(actearly already sees the next act)")
steps_per_tau = round(TAU / mjm.opt.timestep)
for k in (1, 2, 3):
    for _ in range(steps_per_tau):
        mjw.step(m, d)
    act = d.act.numpy()[0]
    print(f"t = {k} tau: act = {act}  exact 1 - e^-{k} = {1 - np.exp(-k):.4f}  "
          f"Euler 1 - (1 - h/tau)^n = {1 - (1 - mjm.opt.timestep / TAU) ** (k * steps_per_tau):.4f}")

## 5. Transmissions and the sparse moment
`d.actuator_moment` is stored as sparse rows per world (`moment_rownnz`, `moment_rowadr`, `moment_colind`).

In [ ]:
# Example 12.5 - ex5_transmissions.py
"""Example 12.5 - transmissions: actuator_length and the sparse actuator moment.

smooth.py::transmission computes, per (world, actuator), the actuator length and its moment arm
(d(length)/d(qpos), one row per actuator). MJWarp stores the rows sparsely, per world:
    d.moment_rownnz[w, i]  non-zeros in row i      d.moment_rowadr[w, i]  where row i starts
    d.moment_colind[w, k]  dof of entry k           d.actuator_moment[w, k] value of entry k
We load test_data/actuation/slidercrank.xml (a slider-crank linkage on hinge 0), add with MjSpec
a site actuator (a thrust along the crank site's x axis, tangent to the crank) and a gear-2 joint
motor on the second hinge, and densify the moment rows at 3 crank angles.
Run: python ex5_transmissions.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

PATH = epath.resource_path("mujoco_warp") / "test_data/actuation/slidercrank.xml"
spec = mujoco.MjSpec.from_file(str(PATH))
spec.add_actuator(name="thrust", trntype=mujoco.mjtTrn.mjTRN_SITE, target="cranksite",
                  gear=[1, 0, 0, 0, 0, 0])
spec.add_actuator(name="geared", trntype=mujoco.mjtTrn.mjTRN_JOINT, target="unactuated",
                  gear=[2, 0, 0, 0, 0, 0])
mjm = spec.compile()
print("actuators:", [(mjm.actuator(i).name or f"#{i}",
                      mujoco.mjtTrn(mjm.actuator_trntype[i]).name) for i in range(mjm.nu)],
      "| nv =", mjm.nv)

ANGLES = [0.0, np.pi / 2, np.pi]
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=len(ANGLES))
qpos = np.zeros((len(ANGLES), mjm.nq), dtype=np.float32)
qpos[:, 0] = ANGLES
d.qpos.assign(qpos)
mjw.forward(m, d)


def dense_moment(d, w, nu, nv):
    """Densify world w's sparse moment rows into an (nu, nv) matrix."""
    nnz, adr = d.moment_rownnz.numpy()[w], d.moment_rowadr.numpy()[w]
    col, val = d.moment_colind.numpy()[w], d.actuator_moment.numpy()[w]
    dense = np.zeros((nu, nv))
    for i in range(nu):
        dense[i, col[adr[i]:adr[i] + nnz[i]]] = val[adr[i]:adr[i] + nnz[i]]
    return dense


print("world 0 sparse layout: rownnz", d.moment_rownnz.numpy()[0], "rowadr",
      d.moment_rowadr.numpy()[0], "colind", d.moment_colind.numpy()[0, :4])
for w, a in enumerate(ANGLES):
    M = dense_moment(d, w, mjm.nu, mjm.nv)
    md = mujoco.MjData(mjm)
    md.qpos[:] = qpos[w]
    mujoco.mj_forward(mjm, md)
    Mc = np.zeros((mjm.nu, mjm.nv))
    mujoco.mju_sparse2dense(Mc, md.actuator_moment, md.moment_rownnz, md.moment_rowadr,
                            md.moment_colind)
    print(f"crank angle {np.degrees(a):5.1f} deg: actuator_length = {d.actuator_length.numpy()[w]}"
          f"  max |moment - C| = {np.abs(M - Mc).max():.1e}")
    print("  dense moment rows (slidercrank, thrust, geared):\n ", M.round(4).tolist())

# sweep the crank for the page figure: length and moment arm of the slider-crank actuator
sweep = np.linspace(-np.pi, np.pi, 61)
ds = mjw.make_data(mjm, nworld=len(sweep))
qs = np.zeros((len(sweep), mjm.nq), dtype=np.float32)
qs[:, 0] = sweep
ds.qpos.assign(qs)
mjw.forward(m, ds)
L = ds.actuator_length.numpy()[:, 0]
arm = [dense_moment(ds, w, mjm.nu, mjm.nv)[0, 0] for w in range(len(sweep))]
fd = np.gradient(L, sweep)
print(f"slider-crank sweep over {len(sweep)} worlds: length in [{L.min():.4f}, {L.max():.4f}], "
      f"max |moment - dL/dq (finite diff)| = {np.abs(np.array(arm) - fd)[1:-1].max():.3f}")

if args.dump_json:
    out = {"angle": [round(float(x), 4) for x in sweep], "length": [round(float(x), 4) for x in L],
           "moment": [round(float(x), 4) for x in arm], "crank": 0.2, "rod": 0.25}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 6. MuJoCo 4 actuators
`nu` (control slots) can be larger than `nactuator`. The DC motor runs in 4.0.1; SO3 and PID actuators are rejected by `put_model`.

In [ ]:
# Example 12.6 - ex6_mujoco4_actuators.py
"""Example 12.6 - MuJoCo 4 actuators: nu controls vs nactuator actuators, the DC motor, SO3.

MuJoCo 4 lets one actuator read several control slots. m.nu counts control slots (the width of
d.ctrl), m.nactuator counts actuators. Actuator i reads ctrl[actuator_ctrladr[i] :
actuator_ctrladr[i] + actuator_ctrlnum[i]], and actuator_ctrlspec[i] is a CtrlInput bit mask
(POS=1, VEL=2, FF=4, VOLTAGE=8) saying what those slots mean.
(a) <dcmotor input="voltage">: one slot, force = (K/R) V - (K^2/R) qdot  (util_misc.dcmotor_*).
    A second dcmotor with input="pos vel" and controller="kp ki kd" is a position servo built on
    the motor model: two slots, so nu > nactuator.
(b) an SO3 ball-joint actuator compiles in MuJoCo 4.0.1 but put_model rejects its transmission.
Run: python ex6_mujoco4_actuators.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=5, suppress=True)


def spec_name(v):
    """Decode actuator_ctrlspec; 0 means a classic actuator that reads one plain ctrl."""
    return mjw.CtrlInput(int(v)).name or "0 (plain ctrl)"


DCMOTOR = """
<mujoco model="dcmotor">
  <option gravity="0 0 0"/>
  <worldbody>
    <body>
      <joint name="joint" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .3 0 0" size=".03" mass="1"/>
    </body>
  </worldbody>
  <actuator>
    <dcmotor name="dcmotor" joint="joint" input="voltage" motorconst="0.05" resistance="2.0"/>
  </actuator>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(DCMOTOR)
print("(a) dcmotor: nu =", mjm.nu, "| nactuator =", mjm.nactuator,
      "| actuator_ctrladr =", mjm.actuator_ctrladr, "| actuator_ctrlnum =", mjm.actuator_ctrlnum,
      "| ctrlspec =", spec_name(mjm.actuator_ctrlspec[0]))
print("    dyn/gain/bias:", mjw.DynType(mjm.actuator_dyntype[0]).name,
      mjw.GainType(mjm.actuator_gaintype[0]).name, mjw.BiasType(mjm.actuator_biastype[0]).name,
      "| gainprm[:2] (R, K) =", mjm.actuator_gainprm[0, :2])
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=3)
d.ctrl.fill_(1.0)                                         # 1 V in every world
d.qvel.assign(np.array([[0.0], [10.0], [20.0]], dtype=np.float32))
mjw.forward(m, d)
R, K = 2.0, 0.05
print("    ctrl = 1 V, qvel = [0, 10, 20]: actuator_force =",
      d.actuator_force.numpy()[:, 0].round(5) + 0.0)
print("    formula (K/R)(V - K qdot) =", np.array([K / R * (1 - K * v) for v in (0, 10, 20)]),
      "<- at qdot = V/K = 20 rad/s the back-EMF cancels the voltage")

SERVO = DCMOTOR.replace(
    "</actuator>",
    '  <dcmotor name="servo" joint="joint" input="pos vel" motorconst="0.05" resistance="2.0"\n'
    '             controller="10 0 1"/>\n    <motor name="extra" joint="joint"/>\n  </actuator>')
mjm2 = mujoco.MjModel.from_xml_string(SERVO)
print("    + servo dcmotor (input='pos vel') + motor: nu =", mjm2.nu, "| nactuator =",
      mjm2.nactuator, "| ctrladr =", mjm2.actuator_ctrladr, "| ctrlnum =",
      mjm2.actuator_ctrlnum)
for i in range(mjm2.nactuator):
    a, n = mjm2.actuator_ctrladr[i], mjm2.actuator_ctrlnum[i]
    print(f"      actuator {i} ({mjm2.actuator(i).name:7s}) reads ctrl[{a}:{a + n}] as",
          spec_name(mjm2.actuator_ctrlspec[i]))
m2 = mjw.put_model(mjm2)
d2 = mjw.make_data(mjm2, nworld=1)
d2.ctrl.assign(np.array([[1.0, 0.3, 0.0, 0.5]], dtype=np.float32))
d2.qpos.fill_(0.1)
d2.qvel.fill_(0.2)
mjw.forward(m2, d2)
print("    ctrl = [1, 0.3, 0, 0.5], q = 0.1, qdot = 0.2: actuator_force =",
      d2.actuator_force.numpy()[0], "| servo formula kp(0.3-q) + kd(0-qdot) =",
      round(10 * (0.3 - 0.1) + 1 * (0.0 - 0.2), 5))

SO3 = """
<mujoco model="so3">
  <worldbody>
    <body>
      <joint name="b" type="ball"/>
      <geom type="capsule" fromto="0 0 0 .3 0 0" size=".03" mass="1"/>
    </body>
  </worldbody>
  <actuator>
    <general joint="b" gaintype="so3" biastype="so3" gainprm="10" biasprm="0 -10 -1"/>
  </actuator>
</mujoco>
"""
mjm3 = mujoco.MjModel.from_xml_string(SO3)
print("(b) SO3 actuator compiles in MuJoCo: actuator_trntype =", mjm3.actuator_trntype,
      mujoco.mjtTrn(mjm3.actuator_trntype[0]).name, "| nu =", mjm3.nu,
      "| nactuator =", mjm3.nactuator)
try:
    mjw.put_model(mjm3)
except NotImplementedError as e:
    print("    mjw.put_model raises NotImplementedError:", e)
print("    mjw.TrnType:", [t.name for t in mjw.TrnType])

In [ ]:
# Example 12.7 - ex7_pid_rejected.py
"""Example 12.7 - MuJoCo 4's <pid> actuator compiles, but MJWarp 4.0.1 rejects it.

io.py::put_model checks every actuator's trntype, dyntype, gaintype and biastype with np.isin
against MJWarp's own enums (types.TrnType, DynType, GainType, BiasType) and raises
NotImplementedError naming the MuJoCo enum values it does not support. A <pid> actuator has
gaintype mjGAIN_PID, which mjw.GainType does not list.
Run: python ex7_pid_rejected.py
"""
import mujoco
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PID = """
<mujoco model="pid">
  <worldbody>
    <body>
      <joint name="j" type="hinge"/>
      <geom type="sphere" size=".1"/>
    </body>
  </worldbody>
  <actuator>
    <pid joint="j" kp="1"/>
  </actuator>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(PID)
print("compiled in mujoco", mujoco.__version__, ": gaintype",
      mujoco.mjtGain(mjm.actuator_gaintype[0]).name, "| biastype",
      mujoco.mjtBias(mjm.actuator_biastype[0]).name, "| nu =", mjm.nu)
try:
    mjw.put_model(mjm)
except NotImplementedError as e:
    print("NotImplementedError:", e)
print("mjw.DynType: ", [t.name for t in mjw.DynType])
print("mjw.GainType:", [t.name for t in mjw.GainType])
print("mjw.BiasType:", [t.name for t in mjw.BiasType])
print("MuJoCo has   ", [n for n in mujoco.mjtGain.__members__], "(mjGAIN_PID is missing in mjw)")

### Actuator damping
An actuator's `damping` attribute is passive: it ends up in `qfrc_damper`.

In [ ]:
# Example 12.7b - ex7b_actuator_damping.py
"""Example 12.7b - actuator damping: an actuator that adds passive damping to its joint.

In MuJoCo 4 an actuator can carry damping="b p1 p2" (m.actuator_damping, m.actuator_dampingpoly).
It is not an actuator force: passive.py::_spring_damper_dof_passive asks
util_misc.actuator_target_damping for every joint and adds gear^2 * (b, p1, p2) of each actuator
that targets the joint to the joint's own damping, so the result lands in qfrc_damper:
    qfrc_damper = -v * (B + P1 |v| + P2 v^2)
    B = dof_damping + gear^2 b,  (P1, P2) = dof_dampingpoly + gear^2 (p1, p2)
m.jnt_actuatorid[j] says which actuator to read: -1 none, >= 0 exactly one, < -1 scan them all.
Run: python ex7b_actuator_damping.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

XML = """
<mujoco model="actuator_damping">
  <option gravity="0 0 0"/>
  <worldbody>
    <body>
      <joint name="j0" type="hinge" axis="0 1 0" damping="0.1"/>
      <geom type="capsule" fromto="0 0 0 .3 0 0" size=".03" mass="1"/>
    </body>
  </worldbody>
  {actuator}
</mujoco>
"""
WITH = '<actuator><general joint="j0" gear="2" damping=".3 .1 .05"/></actuator>'
V = np.array([0.5, -1.0, 2.0], dtype=np.float32)          # one joint velocity per world


def damper(xml):
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=len(V))
    d.qvel.assign(V[:, None])
    mjw.forward(m, d)
    c = []
    for v in V:
        md = mujoco.MjData(mjm)
        md.qvel[0] = v
        mujoco.mj_forward(mjm, md)
        c.append(md.qfrc_damper[0])
    return mjm, d.qfrc_damper.numpy()[:, 0], np.array(c), d.actuator_force.numpy()


mjm0, f0, _, _ = damper(XML.format(actuator=""))
mjm1, f1, c1, af = damper(XML.format(actuator=WITH))
print("jnt_actuatorid without / with actuator:", mjm0.jnt_actuatorid, mjm1.jnt_actuatorid)
print("actuator_damping =", mjm1.actuator_damping, "| actuator_dampingpoly =",
      mjm1.actuator_dampingpoly, "| gear =", mjm1.actuator_gear[0, 0])
gear2 = mjm1.actuator_gear[0, 0] ** 2
B = 0.1 + gear2 * 0.3
P1, P2 = gear2 * 0.1, gear2 * 0.05
by_hand = -V * (B + P1 * np.abs(V) + P2 * V * V)
print("qvel                       ", V)
print("qfrc_damper, joint only    ", f0, "(= -0.1 v)")
print("qfrc_damper, with actuator ", f1)
print(f"by hand: B = 0.1 + 4*0.3 = {B:.2f}, P = 4*(0.1, 0.05) = ({P1:.2f}, {P2:.2f}) ->", by_hand)
print("C MuJoCo qfrc_damper       ", c1)
print("actuator_force (ctrl = 0) ", af[:, 0], "<- the damping is passive, not actuation")

## 7. Muscles

In [ ]:
# Example 12.8 - ex8_muscle.py
"""Example 12.8 - a muscle's force-length curve, one world per length.

test_data/actuation/muscle.xml has a <muscle> on a spatial tendon between a sliding ball and a
world site. A muscle is dyntype MUSCLE + gaintype MUSCLE + biastype MUSCLE:
    force = gain(L, V) * act + bias(L)      (util_misc.muscle_gain, muscle_bias)
gain = -F0 * FL(L) * FV(V) is the active part, bias = -F0 * FP(L) the passive (stretch) part; both
are negative because a muscle pulls (it shortens its tendon). L is the length normalized so that
lengthrange maps onto gainprm range = (0.75, 1.05); F0 = gainprm[2], or scale / acc0 when it is
negative (the default -1). We hold act = 1 and qvel = 0 and sweep the length over 41 worlds.
Run: python ex8_muscle.py [--dump-json PATH] [--dump-png DIR]
"""
import argparse
import json
import os

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
parser.add_argument("--dump-png", default=None)
args, _ = parser.parse_known_args()

PATH = epath.resource_path("mujoco_warp") / "test_data/actuation/muscle.xml"
mjm = mujoco.MjModel.from_xml_path(str(PATH))
prm = mjm.actuator_gainprm[0]
print("dyn/gain/bias:", mujoco.mjtDyn(mjm.actuator_dyntype[0]).name,
      mujoco.mjtGain(mjm.actuator_gaintype[0]).name, mujoco.mjtBias(mjm.actuator_biastype[0]).name)
print("gainprm (range0, range1, force, scale, lmin, lmax, vmax, fpmax, fvmax) =", prm[:9])
print("actuator_lengthrange =", mjm.actuator_lengthrange[0], "| actuator_acc0 =",
      round(float(mjm.actuator_acc0[0]), 4))

# tendon length = distance from the ball's site (at x = q) to the world site at x = 1
NW = 41
lengths = np.linspace(0.02, 0.98, NW)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NW)
d.qpos.assign((1.0 - lengths)[:, None].astype(np.float32))
d.act.fill_(1.0)                       # fully activated muscle
mjw.forward(m, d)
# .copy(): on the CPU device .numpy() is a zero-copy view, so the next forward would overwrite F
L = d.actuator_length.numpy()[:, 0].copy()
F = d.actuator_force.numpy()[:, 0].copy()

# passive part alone: act = 0 leaves only the bias
d.act.zero_()
mjw.forward(m, d)
Fp = d.actuator_force.numpy()[:, 0].copy()

F0 = prm[2] if prm[2] > 0 else prm[3] / mjm.actuator_acc0[0]
print(f"peak force F0 = scale / acc0 = {F0:.4f}")
for i in (4, 12, 20, 28, 36):
    md = mujoco.MjData(mjm)
    md.qpos[0], md.act[0] = 1.0 - lengths[i], 1.0
    mujoco.mj_forward(mjm, md)
    print(f"  tendon length {L[i]:.3f}: force {F[i]:+.4f} (C {md.actuator_force[0]:+.4f}), "
          f"passive part {Fp[i]:+.4f}")
lr = mjm.actuator_lengthrange[0]
L0 = (lr[1] - lr[0]) / (prm[1] - prm[0])                     # length per normalized unit
print(f"active part (F - passive) peaks at length {L[np.argmin(F - Fp)]:.3f}; normalized length 1"
      f" is at {lr[0] + (1 - prm[0]) * L0:.3f}")

if args.dump_json:
    out = {"length": [round(float(x), 4) for x in L], "force": [round(float(x), 4) for x in F],
           "passive": [round(float(x), 4) for x in Fp], "F0": round(float(F0), 4),
           "prm": [round(float(x), 4) for x in prm[:9]],
           "lengthrange": [float(x) for x in mjm.actuator_lengthrange[0]]}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

if args.dump_png:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.plot(L, -F, label="total (act = 1)")
        ax.plot(L, -Fp, "--", label="passive (act = 0)")
        ax.set_xlabel("tendon length [m]")
        ax.set_ylabel("pull = -actuator_force [N]")
        ax.legend()
        fig.tight_layout()
        os.makedirs(args.dump_png, exist_ok=True)
        fig.savefig(os.path.join(args.dump_png, "ex8_muscle.png"), dpi=80)
        print("wrote", os.path.join(args.dump_png, "ex8_muscle.png"))
    except ImportError:
        print("skipped: matplotlib not installed")

In [ ]:
# Force-length curve of the muscle (uses L, F, Fp from Example 12.8).
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 3.2))
    plt.plot(L, -F, label="act = 1 (active + passive)")
    plt.plot(L, -Fp, "--", label="act = 0 (passive only)")
    plt.xlabel("tendon length [m]"); plt.ylabel("pull = -actuator_force [N]"); plt.legend()
    plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

## 8. Tendons and wrapping

In [ ]:
# Example 12.9 - ex9_tendons.py
"""Example 12.9 - tendons: fixed and spatial lengths, the sparse tendon Jacobian, wrapping.

smooth.py::tendon fills d.ten_length (nworld, ntendon) and the tendon Jacobian d.ten_J, stored
sparsely with a pattern fixed at put_model time: row t has m.ten_J_rownnz[t] entries starting at
m.ten_J_rowadr[t], in the dofs m.ten_J_colind[...]; only the values d.ten_J[w, k] change.
  (a) test_data/tendon/fixed.xml: a fixed tendon is a linear combination, L = sum coef_j q_j.
  (b) test_data/tendon/site.xml: spatial tendons through sites, compared with C MuJoCo.
  (c) an elbow: a spatial tendon wraps over a cylinder at the joint (util_misc.wrap). We sweep the
      angle over 31 worlds; once the tendon touches the cylinder its moment arm equals the radius.
Run: python ex9_tendons.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()
TEST = epath.resource_path("mujoco_warp") / "test_data/tendon"


def run_key(path):
    """Load a test model, set keyframe 0, forward in MJWarp and C; return both."""
    mjm = mujoco.MjModel.from_xml_path(str(path))
    md = mujoco.MjData(mjm)
    mujoco.mj_resetDataKeyframe(mjm, md, 0)
    mujoco.mj_forward(mjm, md)
    m = mjw.put_model(mjm)
    d = mjw.put_data(mjm, md, nworld=1)
    mjw.forward(m, d)
    return mjm, md, m, d


def dense_ten_J(m, d, w, ntendon, nv):
    nnz, adr, col = m.ten_J_rownnz.numpy(), m.ten_J_rowadr.numpy(), m.ten_J_colind.numpy()
    J = np.zeros((ntendon, nv))
    for t in range(ntendon):
        J[t, col[adr[t]:adr[t] + nnz[t]]] = d.ten_J.numpy()[w, adr[t]:adr[t] + nnz[t]]
    return J


mjm, md, m, d = run_key(TEST / "fixed.xml")
print("(a) fixed.xml, qpos =", md.qpos, "coef = .25 .5 .75")
print("    ten_length MJWarp", d.ten_length.numpy()[0], "| C", md.ten_length,
      "| by hand", round(.25 * .2 + .5 * .4 + .75 * .6, 4))
print("    ten_J (dense)", dense_ten_J(m, d, 0, mjm.ntendon, mjm.nv)[0],
      "| ten_velocity", d.ten_velocity.numpy()[0], "= J @ qvel")

mjm, md, m, d = run_key(TEST / "site.xml")
print("(b) site.xml: ntendon =", mjm.ntendon, "| ten_J_rownnz =", m.ten_J_rownnz.numpy(),
      "| d.ten_J shape", d.ten_J.shape)
print("    ten_length MJWarp", d.ten_length.numpy()[0])
print("    ten_length C     ", md.ten_length)
Jc = np.zeros((mjm.ntendon, mjm.nv))
mujoco.mju_sparse2dense(Jc, md.ten_J, mjm.ten_J_rownnz, mjm.ten_J_rowadr, mjm.ten_J_colind)
Jw = dense_ten_J(m, d, 0, mjm.ntendon, mjm.nv)
print(f"    max |ten_J MJWarp - C| = {np.abs(Jw - Jc).max():.1e}")

ELBOW = """
<mujoco model="elbow">
  <compiler angle="radian"/>
  <option gravity="0 0 0"/>
  <worldbody>
    <site name="origin" pos="-.3 0 .12"/>
    <geom name="pulley" type="cylinder" size=".06 .05" quat="0.7071068 0.7071068 0 0"
          contype="0" conaffinity="0"/>
    <site name="side" pos="0 0 .2"/>
    <body name="forearm">
      <joint name="elbow" type="hinge" axis="0 1 0" range="-2 2"/>
      <geom type="capsule" fromto="0 0 0 .4 0 0" size=".02" contype="0" conaffinity="0"/>
      <site name="insert" pos=".25 0 .03"/>
    </body>
  </worldbody>
  <tendon>
    <spatial name="flexor">
      <site site="origin"/>
      <geom geom="pulley" sidesite="side"/>
      <site site="insert"/>
    </spatial>
  </tendon>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(ELBOW)
m = mjw.put_model(mjm)
ang = np.linspace(-1.5, 1.5, 31)
d = mjw.make_data(mjm, nworld=len(ang))
d.qpos.assign(ang[:, None].astype(np.float32))
mjw.forward(m, d)
L = d.ten_length.numpy()[:, 0]
J = d.ten_J.numpy()[:, 0]                    # one tendon, one dof: J = dL/dq
wrapnum = d.ten_wrapnum.numpy()[:, 0]
path = d.wrap_xpos.numpy().reshape(len(ang), -1, 3)   # 2 points per wrap object, flattened
print("(c) elbow over a cylinder of radius 0.06, angle sweep over 31 worlds")
for i in (0, 10, 15, 20, 30):
    print(f"    angle {ang[i]:+.2f}: ten_length {L[i]:.4f}  ten_J {J[i]:+.4f}  "
          f"path points {wrapnum[i]} ({'wrapped' if wrapnum[i] == 4 else 'straight'})")
fd = np.gradient(L, ang)
print(f"    max |ten_J - dL/dq (finite diff)| away from the touch point = "
      f"{np.abs(J - fd)[(wrapnum == 4) & (np.roll(wrapnum, 1) == 4)][1:-1].max():.4f}")

if args.dump_json:
    out = {"angle": [round(float(x), 4) for x in ang], "length": [round(float(x), 4) for x in L],
           "J": [round(float(x), 4) for x in J], "wrapnum": wrapnum.tolist(),
           "path": [[[round(float(p[0]), 4), round(float(p[2]), 4)] for p in path[i, :wrapnum[i]]]
                    for i in range(len(ang))],
           "radius": 0.06}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 9. After the solver: rne_postconstraint
`cacc` and `cfrc_ext` are only complete when `rne_postconstraint` runs.

In [ ]:
# Example 12.10 - ex10_postconstraint.py
"""Example 12.10 - cacc and cfrc_ext are only "real" after rne_postconstraint.

During fwd_velocity, smooth.py::rne runs without accelerations (flg_acc=False): d.cacc holds only
the gravity trick (+g upward) plus velocity terms, and d.cfrc_ext is not touched. The contact
force is only known after the solver, so smooth.py::rne_postconstraint (called at the end of
forward) rebuilds cfrc_ext (xfrc_applied + contacts + equality + spatial tendons) and recomputes
cacc with the real qacc. It runs when m.opt.run_rne_postconstraint is True, or when the model has
a sensor that needs it (accelerometer, force, torque, ...: m.sensor_rne_postconstraint).
World 0: a 1 kg ball resting on the floor. World 1: the same ball lifted to z = 1 (free fall).
cacc includes the gravity trick, so after rne_postconstraint it reads like an accelerometer:
+g when resting, 0 in free fall.
Run: python ex10_postconstraint.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=2, nconmax=4, njmax=16)
for _ in range(1000):                       # 2 s: the ball lands and comes to rest
    mjw.step(m, d)
qpos, qvel = d.qpos.numpy().copy(), d.qvel.numpy().copy()
qpos[1, 2], qvel[1] = 1.0, 0.0              # world 1: lift the ball, it starts to fall
d.qpos.assign(qpos)
d.qvel.assign(qvel)
print("ball z per world =", qpos[:, 2].round(4), "| m.sensor_rne_postconstraint =",
      m.sensor_rne_postconstraint)

for flag in (False, True):
    m.opt.run_rne_postconstraint = flag
    d.cfrc_ext.zero_()
    mjw.forward(m, d)
    print(f"run_rne_postconstraint={flag}:")
    for w, name in ((0, "resting"), (1, "falling")):
        print(f"  {name}: cacc[ball] = {d.cacc.numpy()[w, 1]}  cfrc_ext[ball] = "
              f"{d.cfrc_ext.numpy()[w, 1]}")
print("qacc[z] per world =", d.qacc.numpy()[:, 2].round(4), "-> cacc_z = 9.81 + qacc_z;",
      "the floor pushes the resting ball up with m g = 9.81 N")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 10. Gravity compensation and subtree velocities

In [ ]:
# Example 12.11 - ex11_gravcomp_subtree_vel.py
"""Example 12.11 - gravity compensation, and subtree velocities that are computed on demand.

(a) body gravcomp="1" adds an anti-gravity force -gravcomp * m * g at the body's com, mapped to
    joint space by passive.py::_gravity_force into d.qfrc_gravcomp (and then qfrc_passive). The
    body floats. With actuatorgravcomp="true" on a joint the same force is routed into
    qfrc_actuator instead (forward.py::_qfrc_actuator_gravcomp_limits), where actfrcrange clamps it.
(b) d.subtree_linvel / d.subtree_angmom are filled only by smooth.py::subtree_vel, which forward
    calls only when a sensor needs it (m.sensor_subtree_vel). Otherwise call mjw.subtree_vel.
Run: python ex11_gravcomp_subtree_vel.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

FLOAT = """
<mujoco model="float">
  <option timestep="0.002"/>
  <worldbody>
    <body name="balloon" pos="0 0 1" gravcomp="{g}">
      <freejoint/>
      <geom type="sphere" size=".1" mass="0.5"/>
    </body>
  </worldbody>
</mujoco>
"""
print("(a) 0.5 s of simulation, no floor")
for g in ("0", "0.5", "1"):
    mjm = mujoco.MjModel.from_xml_string(FLOAT.format(g=g))
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=1)
    for _ in range(250):
        mjw.step(m, d)
    print(f"  gravcomp={g:3s}: z drift = {d.qpos.numpy()[0, 2] - 1:+.4f} m, "
          f"qfrc_gravcomp[z] = {d.qfrc_gravcomp.numpy()[0, 2]:.4f}, "
          f"qfrc_passive[z] = {d.qfrc_passive.numpy()[0, 2]:.4f}")

ARM = """
<mujoco model="arm">
  <worldbody>
    <body name="arm" gravcomp="1">
      <joint name="j" type="hinge" axis="0 1 0" actuatorgravcomp="{a}" actuatorfrcrange="-1 1"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03" mass="1"/>
    </body>
  </worldbody>
  <actuator><motor joint="j"/></actuator>
</mujoco>
"""
for a in ("false", "true"):
    mjm = mujoco.MjModel.from_xml_string(ARM.format(a=a))
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=1)
    mjw.forward(m, d)
    print(f"  arm, actuatorgravcomp={a:5s}: jnt_actgravcomp={mjm.jnt_actgravcomp[0]} "
          f"qfrc_gravcomp={d.qfrc_gravcomp.numpy()[0, 0]:+.4f} "
          f"qfrc_passive={d.qfrc_passive.numpy()[0, 0]:+.4f} "
          f"qfrc_actuator={d.qfrc_actuator.numpy()[0, 0]:+.4f} (actfrcrange -1..1)")

mjm = mujoco.MjModel.from_xml_string(FLOAT.format(g="0"))
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)
d.qvel.assign(np.array([[1.0, 0.0, 2.0, 0.0, 0.0, 3.0]], dtype=np.float32))
mjw.forward(m, d)
print("(b) m.sensor_subtree_vel =", m.sensor_subtree_vel)
print("  after forward:        subtree_linvel[ball] =", d.subtree_linvel.numpy()[0, 1],
      "subtree_angmom[ball] =", d.subtree_angmom.numpy()[0, 1])
mjw.subtree_vel(m, d)
print("  after mjw.subtree_vel: subtree_linvel[ball] =", d.subtree_linvel.numpy()[0, 1],
      "subtree_angmom[ball] =", d.subtree_angmom.numpy()[0, 1])
print("  expected angmom about the com = I w = (2/5 m r^2) * 3 =", round(0.4 * 0.5 * 0.01 * 3, 4))

## 11. Fluid forces

In [ ]:
# Example 12.12 - ex12_fluid_models.py
"""Example 12.12 - fluid forces: per-world density, inertia box vs ellipsoid, and wind.

passive.py::_fluid_force (one thread per (world, body)) has two models:
  * inertia box (default): the body is replaced by the box with the same inertia; drag is
    0.5 * density * (face area) * |v| v plus a viscous 3 pi diam viscosity v term.
  * ellipsoid (geom fluidshape="ellipsoid"): per-geom blunt and slender drag, Magnus lift,
    Kutta lift, added mass, with coefficients in geom fluidcoef (m.geom_fluid).
The fluid moves with opt.wind: the force depends on the velocity relative to the wind.
(a) 4 worlds with their own opt.density (replace the batched field, Ch 10).
(b) the same ball with the inertia-box model vs the ellipsoid model, density 1.2.
(c) wind = (3, 0, 0) m/s pushes a falling ball sideways.
Run: python ex12_fluid_models.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

NSTEP = 400                                  # 4 s at dt = 0.01

BALL = """
<mujoco model="beach_ball">
  <option timestep="0.01" density="1.2" viscosity="0.00002" wind="{wind}"/>
  <worldbody>
    <body name="ball" pos="0 0 10">
      <freejoint/>
      <geom type="sphere" size=".2" mass="0.1" {shape}/>
    </body>
  </worldbody>
</mujoco>
"""


def fall(xml, density=None, record=False):
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm)
    nworld = len(density) if density is not None else 1
    d = mjw.make_data(mjm, nworld=nworld)
    if density is not None:
        m.opt.density = wp.array(density, dtype=float)     # replace: one density per world
    traj = []
    for i in range(NSTEP):
        mjw.step(m, d)
        if record and i % 10 == 9:
            traj.append([round(float(v), 3) for v in d.qvel.numpy()[:, 2]])
    return mjm, m, d, traj


DENS = [0.0, 1.2, 5.0, 20.0]
mjm, m, d, traj = fall(BALL.format(wind="0 0 0", shape=""), DENS, record=True)
vz = d.qvel.numpy()[:, 2]
print("(a) inertia-box model, m.has_fluid =", m.has_fluid, "| body_fluid_ellipsoid =",
      m.body_fluid_ellipsoid.numpy())
side = np.sqrt(6 * 0.4 * 0.2**2)             # inertia box of a solid sphere: side = sqrt(2.4) r
for rho, v in zip(DENS, vz):
    vt = np.sqrt(2 * 0.1 * 9.81 / (rho * side**2)) if rho else float("inf")
    print(f"  density {rho:5.1f}: vz after {NSTEP * 0.01:.0f} s = {v:+8.3f} m/s"
          f"  (quadratic-drag terminal speed {vt:.3f})")

_, mb, db, _ = fall(BALL.format(wind="0 0 0", shape=""))
_, me, de, _ = fall(BALL.format(wind="0 0 0", shape='fluidshape="ellipsoid"'))
print("(b) density 1.2: inertia box vz =", db.qvel.numpy()[0, 2].round(3),
      "| ellipsoid vz =", de.qvel.numpy()[0, 2].round(3),
      "| body_fluid_ellipsoid =", me.body_fluid_ellipsoid.numpy())
mje = mujoco.MjModel.from_xml_string(BALL.format(wind="0 0 0", shape='fluidshape="ellipsoid"'))
print("    geom_fluid[0] (coef, blunt, slender, ang, kutta, magnus, added-mass...) =",
      mje.geom_fluid[0, :6].round(4))

_, mw, dw, _ = fall(BALL.format(wind="3 0 0", shape=""))
x, vx = dw.qpos.numpy()[0, 0], dw.qvel.numpy()[0, 0]
print(f"(c) wind (3, 0, 0): after {NSTEP * 0.01:.0f} s x drift = {x:+.3f} m, vx = {vx:+.3f} m/s "
      "(approaches the wind speed)")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"density": DENS, "dt": 0.1, "vz": traj}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Vertical velocity over time, one curve per density world (uses traj, DENS from Example 12.12).
try:
    import matplotlib.pyplot as plt
    tr = np.array(traj)
    plt.figure(figsize=(5, 3.2))
    for k, rho in enumerate(DENS):
        plt.plot(np.arange(1, len(tr) + 1) * 0.1, tr[:, k], label=f"density {rho}")
    plt.xlabel("t [s]"); plt.ylabel("vz [m/s]"); plt.legend()
    plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

## Parameter sweep: 1024 servos, one kp each
A batched sweep is the MJWarp way to explore a parameter: every world gets its own `gainprm`/`biasprm`. Each servo is critically damped (`kv = 2 sqrt(kp I)`), so the printed 2% settling time falls as `kp` grows, roughly as `1/sqrt(kp)`. On CPU only 8 worlds run; the numbers are CPU numbers.

In [ ]:
# Parameter sweep: one position servo per world, each with its own kp
# (batched actuator_gainprm / actuator_biasprm, Ch 10). 1024 worlds on a GPU, 8 on CPU.
# Each servo also gets critical velocity damping kv = 2 sqrt(kp * I), so it settles
# without ringing and a stiffer servo settles faster (time constant ~ sqrt(I / kp)).
NW = 1024 if ON_GPU else 8
SERVO = """
<mujoco><option timestep="0.002" gravity="0 0 0"/>
  <worldbody><body><joint name="j" type="hinge" axis="0 1 0" damping="0.05"/>
    <geom type="capsule" fromto="0 0 0 .3 0 0" size=".03" mass="1"/></body></worldbody>
  <actuator><position joint="j" kp="10"/></actuator></mujoco>"""
mjm_s = mujoco.MjModel.from_xml_string(SERVO)
m_s = mjw.put_model(mjm_s, batch_sizes={"actuator_gainprm": NW, "actuator_biasprm": NW})
d_s = mjw.make_data(mjm_s, nworld=NW)
I = float(mjm_s.dof_M0[0])                  # joint-space inertia of the hinge at qpos0
kp = np.geomspace(1, 100, NW)
kv = 2 * np.sqrt(kp * I)                     # critical damping
g, b = m_s.actuator_gainprm.numpy(), m_s.actuator_biasprm.numpy()
g[:, 0, 0] = kp           # gain = kp
b[:, 0, 1] = -kp          # bias = -kp * length ...
b[:, 0, 2] = -kv          #      - kv * velocity
m_s.actuator_gainprm.assign(g)
m_s.actuator_biasprm.assign(b)
d_s.ctrl.fill_(0.5)       # every servo aims at 0.5 rad
traj_s = []
for i in range(500):
    mjw.step(m_s, d_s)
    if (i + 1) % 10 == 0:                    # samples at t = 0.02, 0.04, ..., 1.00 s
        traj_s.append(d_s.qpos.numpy()[:, 0].copy())   # .copy(): on CPU .numpy() is a view
traj_s = np.array(traj_s)
t = (np.arange(len(traj_s)) + 1) * 10 * mjm_s.opt.timestep
err = np.abs(traj_s - 0.5)
band = 0.02 * 0.5                            # 2% of the 0.5 rad step
def settling_time(e):
    if e[-1] > band:
        return float("nan")                  # not settled by the last sample
    out = np.nonzero(e > band)[0]
    return float(t[out.max() + 1]) if len(out) else float(t[0])
settle = [settling_time(e) for e in err.T]
print(f"I = {I:.4f} kg m^2, sample times {t[0]:.2f} ... {t[-1]:.2f} s")
print("kp:              ", [round(float(x), 1) for x in kp[:8]])
print(f"q at t={t[-1]:.2f} s:   ", [round(float(x), 3) for x in traj_s[-1, :8]], "(target 0.5)")
print("2% settling [s]: ", [round(x, 2) for x in settle[:8]], "(nan: not settled by 1 s)")
try:
    import matplotlib.pyplot as plt
    for k in np.linspace(0, NW - 1, 6).astype(int):
        plt.plot(t, traj_s[:, k], label=f"kp={kp[k]:.1f}")
    plt.axhline(0.5, color="k", ls="--", lw=1)
    plt.xlabel("t [s]"); plt.ylabel("q [rad]"); plt.legend(); plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

## Exercise 12.1: a PD servo from a general actuator
Fill in `gainprm` and `biasprm` so that `actuator_force = KP (ctrl - q) - KD qdot`.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

KP, KD = 30.0, 2.0
SERVO = """
<mujoco>
  <option gravity="0 0 0"/>
  <worldbody>
    <body>
      <joint name="j" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .3 0 0" size=".03" mass="1"/>
    </body>
  </worldbody>
  <actuator>
    <general joint="j" gaintype="fixed" gainprm="..." biastype="affine" biasprm="..."/>
  </actuator>
</mujoco>
"""
# TODO: fill in gainprm and biasprm so that force = KP * (ctrl - q) - KD * qdot
mjm = mujoco.MjModel.from_xml_string(SERVO)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4)
rng = np.random.default_rng(0)
q, v, u = (rng.uniform(-1, 1, (4, 1)).astype(np.float32) for _ in range(3))
d.qpos.assign(q); d.qvel.assign(v); d.ctrl.assign(u)
mjw.forward(m, d)
force = d.actuator_force.numpy()
assert np.allclose(force, KP * (u - q) - KD * v, atol=1e-5), force
print("PD servo OK:", force[:, 0])

In [ ]:
#@title Exercise 12.1: solution (double-click to show the code)

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

KP, KD = 30.0, 2.0
SERVO = f"""
<mujoco>
  <option gravity="0 0 0"/>
  <worldbody>
    <body>
      <joint name="j" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .3 0 0" size=".03" mass="1"/>
    </body>
  </worldbody>
  <actuator>
    <general joint="j" gaintype="fixed" gainprm="{KP}" biastype="affine" biasprm="0 {-KP} {-KD}"/>
  </actuator>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(SERVO)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4)
rng = np.random.default_rng(0)
q, v, u = (rng.uniform(-1, 1, (4, 1)).astype(np.float32) for _ in range(3))
d.qpos.assign(q); d.qvel.assign(v); d.ctrl.assign(u)
mjw.forward(m, d)
force = d.actuator_force.numpy()
assert np.allclose(force, KP * (u - q) - KD * v, atol=1e-5), force
print("PD servo OK:", force[:, 0])

## Exercise 12.2: the slider-crank moment arm by finite differences
The moment row is d(actuator_length)/d(qpos). Check it with a central difference, using one world per sample.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

PATH = epath.resource_path("mujoco_warp") / "test_data/actuation/slidercrank.xml"
mjm = mujoco.MjModel.from_xml_path(str(PATH))
m = mjw.put_model(mjm)
q0, eps = 0.7, 1e-3
# TODO: 3 worlds at crank angles q0, q0 + eps, q0 - eps; one forward; central difference of
# actuator_length; read the slider-crank moment on dof 0 from the sparse moment arrays.
fd = ...
moment = ...
assert abs(moment - fd) < 1e-3, (moment, fd)

In [ ]:
#@title Exercise 12.2: solution (double-click to show the code)

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

PATH = epath.resource_path("mujoco_warp") / "test_data/actuation/slidercrank.xml"
mjm = mujoco.MjModel.from_xml_path(str(PATH))
m = mjw.put_model(mjm)
q0, eps = 0.7, 1e-3
d = mjw.make_data(mjm, nworld=3)
qpos = np.zeros((3, mjm.nq), dtype=np.float32)
qpos[:, 0] = [q0, q0 + eps, q0 - eps]
d.qpos.assign(qpos)
mjw.forward(m, d)
L = d.actuator_length.numpy()[:, 0]
fd = (L[1] - L[2]) / (2 * eps)
adr = d.moment_rowadr.numpy()[0, 0]
nnz = d.moment_rownnz.numpy()[0, 0]
cols = d.moment_colind.numpy()[0, adr:adr + nnz]
vals = d.actuator_moment.numpy()[0, adr:adr + nnz]
moment = dict(zip(cols.tolist(), vals.tolist())).get(0, 0.0)
print(f"moment {moment:.5f}  finite difference {fd:.5f}")
assert abs(moment - fd) < 1e-3, (moment, fd)

## Exercise 12.3: a fixed-tendon spring
Predict `qfrc_spring` of a stretched fixed tendon from its length, its Jacobian and the spring law.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

TENDON = """
<mujoco>
  <option gravity="0 0 0"/>
  <worldbody>
    <body><joint name="a" type="slide" axis="1 0 0"/><geom type="sphere" size=".05" mass="1"/>
      <body><joint name="b" type="slide" axis="1 0 0"/><geom type="sphere" size=".05" mass="1"/></body>
    </body>
  </worldbody>
  <tendon>
    <fixed name="t" stiffness="50" springlength="0.1">
      <joint joint="a" coef="1"/><joint joint="b" coef="-2"/>
    </fixed>
  </tendon>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(TENDON)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)
d.qpos.assign(np.array([[0.4, 0.05]], dtype=np.float32))
mjw.forward(m, d)
expected = ...   # TODO: qfrc_spring predicted from L, the tendon Jacobian and the spring law
assert np.allclose(d.qfrc_spring.numpy()[0], expected, atol=1e-5), d.qfrc_spring.numpy()

In [ ]:
#@title Exercise 12.3: solution (double-click to show the code)

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

TENDON = """
<mujoco>
  <option gravity="0 0 0"/>
  <worldbody>
    <body><joint name="a" type="slide" axis="1 0 0"/><geom type="sphere" size=".05" mass="1"/>
      <body><joint name="b" type="slide" axis="1 0 0"/><geom type="sphere" size=".05" mass="1"/></body>
    </body>
  </worldbody>
  <tendon>
    <fixed name="t" stiffness="50" springlength="0.1">
      <joint joint="a" coef="1"/><joint joint="b" coef="-2"/>
    </fixed>
  </tendon>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(TENDON)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)
d.qpos.assign(np.array([[0.4, 0.05]], dtype=np.float32))
mjw.forward(m, d)
L = 0.4 - 2 * 0.05                     # fixed tendon: sum coef * q
J = np.array([1.0, -2.0])              # dL/dq = the coefficients
expected = J * (-50 * (L - 0.1))       # tendon force -k (L - springlength), mapped by J^T
print(d.ten_length.numpy(), d.qfrc_spring.numpy(), expected)
assert np.allclose(d.qfrc_spring.numpy()[0], expected, atol=1e-5), d.qfrc_spring.numpy()

## Stretch: your own passive force through `m.callback.passive`
Write a kernel that adds a drag `-c * qvel` to `d.qfrc_passive` and register it as `m.callback.passive` (recall Ch 8; the callback catalogue is Ch 26). The sketch below shows one way.

In [ ]:
# Stretch sketch: a custom linear drag through m.callback.passive (runs at the end of passive.py::passive).
C = 0.8

@wp.kernel
def drag(qvel: wp.array2d(dtype=float), c: float, qfrc_passive: wp.array2d(dtype=float)):
    worldid, dofid = wp.tid()
    qfrc_passive[worldid, dofid] -= c * qvel[worldid, dofid]

def my_passive(m, d):
    wp.launch(drag, dim=(d.nworld, m.nv), inputs=[d.qvel, C], outputs=[d.qfrc_passive])

XML = """<mujoco><option timestep="0.01"/><worldbody><body pos="0 0 1"><freejoint/>
<geom type="sphere" size=".1" mass="1"/></body></worldbody></mujoco>"""
mjm_c = mujoco.MjModel.from_xml_string(XML)
m_c = mjw.put_model(mjm_c)
m_c.callback.passive = my_passive
d_c = mjw.make_data(mjm_c, nworld=2)
for _ in range(800):
    mjw.step(m_c, d_c)
print("vz after 8 s:", d_c.qvel.numpy()[:, 2], "| terminal -m g / c =", -9.81 / C)